# Code Annotator

Evaluate how different models execute Annotating a python code


In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set")

In [ ]:
openai_client = OpenAI(api_key=openai_api_key)

openrouter_client = OpenAI(
    api_key=openrouter_api_key,
    base_url="https://openrouter.ai/api/v1"
)

models = [
    "gpt-4o-mini",
    "gpt-4o",
    "qwen/qwen3-coder-30b-a3b-instruct",
    "meta-llama/llama-3.3-70b-instruct",
]

clients = {
    "gpt-4o-mini": openai_client,
    "gpt-4o": openai_client                          ,
    "qwen/qwen3-coder-30b-a3b-instruct": openrouter_client,
    "meta-llama/llama-3.3-70b-instruct": openrouter_client,
}

print(f"{len(models)} models ready")

In [ ]:
system_prompt = """You are an expert Python developer.
Your task is to add clear, accurate docstrings and inline comments to Python code.
Rules:
- Add a docstring to every function describing what it does, its parameters, and what it returns
- Add inline comments to any non-obvious lines of code
- Do NOT change the logic or behaviour of the code in any way
- Respond only with the complete annotated Python code, no explanations
"""

def user_prompt_for(python_code):
    return f"""Add docstrings and comments to this Python code.
Do not change any logic. Return only the annotated Python code.
```python
{python_code}
```
"""

def messages_for(python_code):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt_for(python_code)}
    ]

print("Prompts ready")

In [ ]:
def add_docstrings(model, python_code):
    client = clients[model]
    
    response = client.chat.completions.create(
        model=model,
        messages=messages_for(python_code)
    )
    
    reply = response.choices[0].message.content
    # Strip markdown fences if the model added them despite instructions
    reply = reply.replace('```python', '').replace('```', '')
    return reply.strip()

print("Function ready")

In [ ]:
test_code = """
def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result
"""

result = add_docstrings("gpt-4o-mini", test_code)
print(result)

In [ ]:
with gr.Blocks(theme=gr.themes.Monochrome(), title="Docstring Generator") as ui:
    
    # Row 1: Code panels side by side
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            input_code = gr.Code(
                label="Python code (paste yours here)",
                language="python",
                lines=24
            )
        with gr.Column(scale=6):
            output_code = gr.Code(
                label="Annotated code (with docstrings + comments)",
                language="python",
                lines=24
            )
    
    # Row 2: Controls
    with gr.Row():
        model_dropdown = gr.Dropdown(
            models,
            value=models[0],
            label="Select model"
        )
        generate_btn = gr.Button("✨ Add Docstrings", variant="primary")
    
    # Wire button to function
    generate_btn.click(
        fn=add_docstrings,
        inputs=[model_dropdown, input_code],
        outputs=[output_code]
    )

ui.launch(inbrowser=True)